# Train YOLO11n on Stamp/Signature/QR Dataset

This notebook trains YOLO11n for object detection


In [18]:
import os, torch
os.environ['CUDA_VISIBLE_DEVICES'] = os.environ.get('CUDA_VISIBLE_DEVICES', '0')
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    try:
        print('Current device index:', torch.cuda.current_device())
        for i in range(torch.cuda.device_count()):
            print(f'GPU {i}:', torch.cuda.get_device_name(i))
    except Exception as e:
        print('CUDA query error:', e)
else:
    print('Falling back to CPU')

CUDA available: True
CUDA device count: 1
Current device index: 0
GPU 0: NVIDIA GeForce RTX 4090


In [19]:
from pathlib import Path
from ultralytics import YOLO

DATA_YAML = '/home/sara_team/Desktop/case/IDP_stamp_signature_detection.v5i.yolov11/data.yaml'
assert Path(DATA_YAML).exists(), f'Not found: {DATA_YAML}'
print('Using data.yaml:', DATA_YAML)

Using data.yaml: /home/sara_team/Desktop/case/IDP_stamp_signature_detection.v5i.yolov11/data.yaml


In [20]:
model = YOLO('yolo11n.pt')

# Training configuration
train_cfg = dict(
    data=DATA_YAML,
    imgsz=640,
    epochs=100,
    batch=-1,            # auto-batch to avoid OOM
    device='0' if torch.cuda.is_available() else 'cpu',
    workers=6,
    amp=True,            # mixed precision
    patience=20,         # early stopping
    lr0=0.01,            # slightly lower LR can help reduce overfitting
    weight_decay=0.0005,
    cls=1.0,             # increase classification loss weigh
    label_smoothing=0.05,
    mosaic=0.5,          # default ~0.5; keep moderate
    mixup=0.0,           # avoid unrealistic mixing for signatures
    copy_paste=0.0,      # avoid pasted instances for signatures
    close_mosaic=10,     # disable mosaic in final N epochs to stabilize
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.5, shear=1.0,
    name='yolo11n_idp_qr',
    project='runs/train'
)

results = model.train(**train_cfg)
print(results)

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.228 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24069MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/sara_team/Desktop/case/IDP_stamp_signature_detection.v5i.yolov11/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.5, multi_scale=False, n

## Validate and visualize

Runs standard validation and saves plots (PR curves, confusion matrix) in the run directory.


In [12]:
# Validate on the validation split defined in data.yaml
metrics = model.val(data=DATA_YAML, imgsz=640, split='val')
print(metrics)

Ultralytics 8.3.228 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24069MiB)
YOLO11n summary (fused): 100 layers, 2,582,932 parameters, 0 gradients, 6.3 GFLOPs
YOLO11n summary (fused): 100 layers, 2,582,932 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4576.8±2427.4 MB/s, size: 186.6 KB)
val: Scanning /home/sara_team/Desktop/case/IDP_stamp_signature_detection.v4i.yolov11/valid_with_qr/labels.cache... 329 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 329/329 1.7Mit/s 0.0s0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4576.8±2427.4 MB/s, size: 186.6 KB)
val: Scanning /home/sara_team/Desktop/case/IDP_stamp_signature_detection.v4i.yolov11/valid_with_qr/labels.cache... 329 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 329/329 1.7Mit/s 0.0s0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 8.9it/s 2.4s<0.3s
                 Class     Images  Instance